# Assignment 4 — What Your Strategy Is Made Of, and How Much of It to Hold


Graded on completion. Groups of three; each of you submits your own copy.

---

**Group members**

- Name / NYU email:
- Name / NYU email:
- Name / NYU email:

---

Two parts, both on your group's own strategy.

**Part 1** takes it apart: how much of its risk is factor exposure, and how much
is its own.

**Part 2** puts it to work. You already hold the market — how much of your
strategy should you add? There is a formula for that, and three cruder rules. You
will run all four through the estimate → tune → test process from *Backtesting*
and find out which one you would actually have wanted.

## 🛠️ Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 4]
import warnings; warnings.filterwarnings('ignore')

BASE  = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")

# The Fama-French five factors plus momentum (UMD) -- the same file the lectures use
FF  = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True).loc['1980-01-31':'2000-12-31']
SIX = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)       # annualized, from monthly returns

print("ready")

### Your strategy

Rebuild your long-short series from Assignment 2 — same convention: **NYSE
breakpoints, value-weighted, D10−D1**, with the signal and the weights lagged one
month and the return you earn being this month's `ret`. Call it `ls`.

The last line puts your strategy and the factors in one table, `j`, on the same
months. Use `j` for everything below.

In [ ]:
MY_SIGNAL = "____"        # ← the same signal as Assignments 2 and 3

# rebuild your long-short (reuse your Assignment 2 code)
ls = ____

j = pd.concat([ls.rename('ls'), FF], axis=1).dropna()
print(f"{MY_SIGNAL}: {len(j)} months, raw return {j.ls.mean()*12:+.2%}/yr, Sharpe {sharpe(j.ls):.2f}")

---

# Part 1 — What your strategy is made of

### Q1 — Against the market

Regress your long-short on `Mkt-RF`, with an intercept, over the full sample. The
regression splits the variance of your return into two pieces that add up:

$$\operatorname{var}(r) \;=\; \underbrace{\beta^2 \operatorname{var}(r_m)}_{\text{factor}} \;+\; \underbrace{\operatorname{var}(\varepsilon)}_{\text{strategy-specific}}$$

Report β, and the **share** of your strategy's variance that sits in each piece.

> **✅ Check.** The two shares add to one, and the factor share is the
> regression's R².

In [ ]:
# your code here

**β:** · **factor share:** · **strategy-specific share:**

### Q2 — Against six factors

Now the same split with all six factors. Build the two pieces the way
*Multi-Factor Models* did for Berkshire and ARKK:

- **tracking piece** — each factor times its loading, added up across the factors
- **hedged piece** — your strategy minus the tracking piece

Report the variance share of each piece, and the annualized Sharpe ratio of three
things: your strategy, the tracking piece, and the hedged piece.

> **(a)** How much of your strategy's risk could you have bought as factor
> exposure? Compare with Q1: what did the other five factors pick up that the
> market missed?
>
> **(b)** Which piece carries the Sharpe ratio? Say what that tells you about
> what your group built.

In [ ]:
# your code here

**(a)**

**(b)**

---

# Part 2 — How much of it to hold

You already hold the market. You have a strategy with some alpha. How much of it
should you add?

### The formula

First **orthogonalize**: take the market out of your strategy, so that the two
things you are combining are uncorrelated.

$$h_t = r_t - \beta\, r_{m,t}$$

`h` is your hedged strategy. Its mean is α, its volatility is σ_ε, and its
correlation with the market is zero. For two uncorrelated assets, the portfolio
with the highest Sharpe ratio holds each one in proportion to its **mean over its
variance**:

$$w_m \;\propto\; \frac{\mu_m}{\sigma_m^2} \qquad\qquad w_h \;\propto\; \frac{\alpha}{\sigma_\varepsilon^2}$$

That is the rule from class. It needs four numbers — μ_m, σ_m, α and σ_ε — and
you have to estimate every one of them.

### Three cruder rules

| rule | weight on the market | weight on the hedged strategy |
|---|---|---|
| **Optimal** | $\mu_m / \sigma_m^2$ | $\alpha / \sigma_\varepsilon^2$ |
| **Equal** | 1 | 1 |
| **Inverse volatility** | $1 / \sigma_m$ | $1 / \sigma_\varepsilon$ |
| **Appraisal** | $\mu_m / \sigma_m$ | $\alpha / \sigma_\varepsilon$ |

Only the proportions matter, so rescale each pair so the absolute values add to
one: divide both weights by $|w_m| + |w_h|$.

> **📌 A negative α gives a negative weight.** The rule is telling you to short
> your strategy. That is not a bug — keep the sign.

### The three samples

The same split as in *Backtesting*.

| sample | months | its job |
|---|---|---|
| **Estimate** | 1980–1989 | β, and the four numbers |
| **Tune** | 1990–1994 | choose the rule |
| **Test** | 1995–2000 | report — once, at the end |

In [ ]:
EST = j.loc[:'1989-12-31']
TUN = j.loc['1990-01-01':'1994-12-31']
TST = j.loc['1995-01-01':]

print(f"ESTIMATE {EST.index[0]:%Y-%m} to {EST.index[-1]:%Y-%m}  ({len(EST)} months)")
print(f"TUNE     {TUN.index[0]:%Y-%m} to {TUN.index[-1]:%Y-%m}  ({len(TUN)} months)")
print(f"TEST     {TST.index[0]:%Y-%m} to {TST.index[-1]:%Y-%m}  ({len(TST)} months)")

### Q3 — Estimate

Using the **estimate sample only**:

- regress your strategy on the market to get α, β and σ_ε — the standard
  deviation of the residual
- compute the market's mean μ_m and standard deviation σ_m

Then build the hedged strategy `h` for **every** month in `j`, using that one β.

> **✅ Check.** In the estimate sample, `h` and the market have a correlation of
> zero. Compute it in the tune and test samples too.

In [ ]:
# your code here

**α (annualized):** · **β:** · **σ_ε (annualized):** · **μ_m (annualized):** · **σ_m (annualized):**

**corr(h, market) — estimate:** · **tune:** · **test:**

**Why is it not zero in the later samples, and what does that do to the "hedge"?**

### Q4 — The four portfolios

From your estimate-sample numbers, compute the four pairs of weights and rescale
them. Then build each portfolio's monthly return over the whole of `j`:

$$r_{p,t} = w_m\, r_{m,t} + w_h\, h_t$$

> **✅ Check.** On the estimate sample, the optimal rule's Sharpe ratio equals
> $\sqrt{SR_m^2 + AR^2}$ — the formula from *Performance Evaluation* — where
> $AR = \alpha/\sigma_\varepsilon$, annualized. If yours does not match to three
> decimals, find out why before going on.
>
> No other rule can beat it in that sample. Why not?

> **📌 Two rules giving almost the same weights is not a bug.** If it happens to
> you, work out what must be true of your four numbers for those two rules to
> agree. You will want that for Q7.

In [ ]:
# your code here

**Weights (market, hedged strategy) — optimal:** · **equal:** · **inverse vol:** · **appraisal:**

**Why can no rule beat the optimal one in the estimate sample?**

### Q5 — Tune

> **🤔 Predict first.** Which of the four rules will have the highest Sharpe ratio
> in the tune sample? Write it down before you run anything.

Compute the Sharpe ratio of each of the four portfolios on the **tune sample**.
The rule with the highest one is the rule your process picks.

In [ ]:
# your code here

**Predicted:** · **The tune sample picks:**

### Q6 — Test

Now, and only now, the test sample. One table of Sharpe ratios: the four rules,
the market alone, and your strategy alone. Mark the rule the tune sample chose.

> **(a)** Did the tuned choice win? Did the optimal rule?
>
> **(b)** Compare your estimate-sample α with what the hedged strategy actually
> earned, per year, in the test sample. Of the four numbers the optimal rule
> needs, which one moved the most between the estimate sample and the test
> sample?

In [ ]:
# your code here

**(a)**

**(b)**

### Q7 — When is a cruder rule the better rule?

The optimal rule is optimal **if** μ_m, σ_m, α and σ_ε are the true values. You
estimated them from ten years of monthly returns.

Each cruder rule is what the optimal rule turns into when you stop trusting one
of its inputs. 

- Equal weight: Peak naivete--assumes it is all the same and god for diversification
- 1/vol : this implictly assume that the strategies are uncorrelated and have the same Sharpe-Ratio
- Sharpe-ratio: Implicitly assume their volatilties are the same and the strategies are uncorrelated.
- Appraisal-ratio: the optimal one--assuming they are indeed uncorrelated

**Equal:**

**Inverse volatility:**

**Appraisal:**

### Q8 — The memo

> **📝 Maximum ten sentences.**
>
> Your PM holds the market and asks one thing: **should we add your strategy, and
> how much of it?**
>
> - Say what your strategy is made of — how much of its risk is exposure they
>   could have bought, and how much is its own.
> - Give the weight each rule puts on it, and say which rule you would run from
>   here.
> - Say what the test sample showed, and how much of your estimate-sample alpha
>   turned up in it.
> - Name the one number, of the four, that you trust least — and what you would
>   do about it.

**Memo:**